# Loop SLM — GPU Smoke Test
This notebook runs a small, one-step hardware and correctness check. It does **not** train a model or save model weights.

**Before running:** in Kaggle, set Accelerator to GPU. Then use Run All.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path

assert os.path.exists('/kaggle/working'), 'This notebook is intended for Kaggle.'
import torch
assert torch.cuda.is_available(), 'GPU is not enabled. In Notebook options, set Accelerator = GPU, then restart and Run All.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
REPO_URL = 'https://github.com/tanushgargjanuary-bot/SLM-Training.git'
REPO_DIR = Path('/kaggle/working/SLM-Training')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
%cd /kaggle/working/SLM-Training
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)


In [ ]:
report_path = '/kaggle/working/smoke-report.json'
subprocess.run([
    sys.executable, 'scripts/cloud_smoke_test.py',
    '--sequence-length', '64', '--loops', '1', '2',
    '--output', report_path,
], check=True)
report = json.loads(Path(report_path).read_text())
print(json.dumps(report, indent=2))


## What to check
- Both variants finish without CUDA out-of-memory errors.
- Loss is finite.
- Parameter counts are identical.
- Loop-2 uses more compute/memory than Loop-1; the measured difference is our initial hardware baseline.

Download `smoke-report.json` from the Kaggle output files after the run. This report is diagnostic only, not benchmark evidence.